In [9]:
from data_frame.optimization.caching.cache_manager import CacheManager
from data_frame.spark_utils import get_spark
import time
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

In [7]:
spark = get_spark(app_name="Cache manager")

## 1. Basic Caching

In [8]:
# Create DataFrame with expensive transformation
df_large = spark.range(0, 10000000) \
    .withColumn("value", F.rand()) \
    .withColumn("value_squared", F.col("value") ** 2)

# Cache the DataFrame
df_cached = CacheManager.cache_dataframe(df_large)
print(f"DataFrame cached: {CacheManager.check_cached(spark, df_cached)}")

# Time first action (without cache benefit for first run)
start = time.time()
count1 = df_cached.count()
time1 = time.time() - start

# Time second action (should use cache)
start = time.time()
count2 = df_cached.count()
time2 = time.time() - start

print(f"First count: {count1}, Time: {time1:.2f}s")
print(f"Second count: {count2}, Time: {time2:.2f}s")
print(f"Speed improvement: {time1/time2:.1f}x")

DataFrame cached: True


First count: 10000000, Time: 3.69s
Second count: 10000000, Time: 0.28s
Speed improvement: 13.2x


In [11]:
# Create DataFrame
df_test = spark.range(0, 1000000) \
    .withColumn("random", F.rand()) \
    .withColumn("id_mod", F.col("id") % 10)

# Cache with different storage levels
storage_levels = {
    "Memory Only": StorageLevel.MEMORY_ONLY,
    "Memory and Disk": StorageLevel.MEMORY_AND_DISK,
    "Memory Only Ser": StorageLevel.MEMORY_AND_DISK_DESER,
    "Disk Only": StorageLevel.DISK_ONLY
}

for name, level in storage_levels.items():
    df_persist = CacheManager.cache_dataframe(df_test, level)
    # Force caching
    df_persist.count()
    print(f"\n{name}:")
    print(f"  Storage Level: {df_persist.storageLevel}")
    
    # Time query
    start = time.time()
    result = df_persist.groupBy("id_mod").agg(F.avg("random")).collect()
    query_time = time.time() - start
    print(f"  Query time: {query_time:.3f}s")
    
    df_persist.unpersist()


Memory Only:
  Storage Level: Memory Serialized 1x Replicated
  Query time: 1.054s

Memory and Disk:
  Storage Level: Disk Memory Serialized 1x Replicated
  Query time: 0.260s

Memory Only Ser:
  Storage Level: Disk Memory Deserialized 1x Replicated
  Query time: 0.223s

Disk Only:
  Storage Level: Disk Serialized 1x Replicated
  Query time: 0.191s


## 3. Uncache and Cleanup

In [12]:
# Cache multiple DataFrames
df1 = spark.range(1000000).cache()
df2 = spark.range(1000000).withColumn("x", F.rand()).cache()

# Force caching
df1.count()
df2.count()

print(f"df1 cached: {CacheManager.check_cached(spark, df1)}")
print(f"df2 cached: {CacheManager.check_cached(spark, df2)}")

# Uncache specific DataFrame
CacheManager.uncache_dataframe(df1)
print(f"\nAfter uncaching df1: {CacheManager.check_cached(spark, df1)}")

# Clear all cache
CacheManager.clear_all_cache(spark)
print(f"After clear all: df2 cached: {CacheManager.check_cached(spark, df2)}")

df1 cached: True
df2 cached: True

After uncaching df1: False
After clear all: df2 cached: True
